In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import sys
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
def summary(dataset, target):
    print("Dimensions: ", dataset.shape)
    print("Datatypes: ", dataset.dtypes.unique())
    print("Missing values:", dataset.isna().sum().sum())
    print("Missing values total:", dataset.isna().sum().sum())

    if dataset[target].isna().sum() > 0:
        dataset = dataset.dropna(subset=[target]).reset_index(drop=True)
        print("Target had missing values. These rows were removed.")
        print("Dimensions: ", dataset.shape)

    print("Target_table:")
    print(dataset[target].value_counts(dropna=False))
    print(dataset[target].value_counts(normalize=True, dropna=False))

In [ ]:
na_values = ["NA", "", "NULL", "unknown", "Unknown", "na"]

# DIABETES #

In [ ]:
diabetes = pd.read_csv("../datasets/diabetes.csv", na_values=na_values)

In [ ]:
print("---------------------------------------")
print("Dataset preview")
print("---------------------------------------")
print(diabetes.head())
print("---------------------------------------")
print("Columns types")
print("---------------------------------------")
print(diabetes.dtypes)
print("---------------------------------------")
print("Summary")
print("---------------------------------------")
summary(diabetes, "Outcome")

In [ ]:
plt.figure(figsize=(4, 3))
diabetes['Outcome'].value_counts(normalize=True).plot(kind="bar")
plt.show()

In [ ]:
numeric_vars = diabetes.columns[:-1]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for ax, col in zip(axes, numeric_vars):
    ax.hist(diabetes[col], bins=20)
    ax.set_title(col)
for ax in axes[len(numeric_vars):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
corr = diabetes.corr()

plt.figure(figsize=(6, 5))
cmap = plt.get_cmap("RdBu")   
sns.heatmap(
    corr,
    cmap=cmap,
    vmin=-1, vmax=1,
    center=0,                 # ensures white at 0
    square=True,
    annot=True, fmt=".2f",
    annot_kws={"size": 7},
    linewidths=0.5,
    cbar_kws={"orientation": "vertical"}
)
plt.title("Pearson Correlation (Numerical Variables)")
plt.xticks(rotation=30, ha="right", fontsize=8, color="black")
plt.yticks(rotation=0, fontsize=8, color="black")

plt.tight_layout()
plt.show()

In [ ]:
diabetes_train, diabetes_test = train_test_split(diabetes, test_size=0.2, random_state=42)
print(diabetes_train.shape)
print(diabetes_test.shape)

While no values are marked as NA, we treat the number 0 as missing variable in all columns except Pregnancy, DiabetesPedigreeFunction and Outcome. For all other variables 0 is an impossible value.

In [ ]:
rep_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "Age"]

# Replace impossible values with NaN
diabetes_train[rep_cols] = diabetes_train[rep_cols].replace(0, np.nan)
diabetes_test[rep_cols] = diabetes_test[rep_cols].replace(0, np.nan)

print("Missing values:", diabetes_train.isna().sum())
print("Missing values:", diabetes_test.isna().sum())

# imputer = SimpleImputer(strategy="median")
# diabetes_train[rep_cols] = imputer.fit_transform(diabetes_train[rep_cols])
imputer = KNNImputer(n_neighbors=5)
diabetes_train[rep_cols] = imputer.fit_transform(diabetes_train[rep_cols])
diabetes_test[rep_cols] = imputer.transform(diabetes_test[rep_cols])

print("Missing values:", diabetes_train.isna().sum().sum())
print("Missing values:", diabetes_test.isna().sum().sum())


In [ ]:
diabetes_train.to_csv("../datasets/diabetes_train.csv", index=False)
diabetes_test.to_csv("../datasets/diabetes_test.csv", index=False)

## Let's try to make the dataset more balanced by oversampling
We already have a small dataset, so undersampling might reduce it too much

In [ ]:
print("----------------------------------------------------------------------------------------------")
print("Oversampling using RandomOverSampler")
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(diabetes_train.drop(['Outcome'], axis = 1), diabetes_train['Outcome'])
print(f"Original class distribution: {diabetes_train['Outcome'].value_counts()}")
print(f"Resampled class distribution: {pd.Series(y_train_resampled).value_counts()}")

diabetes_train_resampled = pd.concat(
    [X_train_resampled, y_train_resampled],
    axis=1
)
diabetes_train_resampled.to_csv("../datasets/diabetes_train_balanced.csv", index=False)

# DRY BEANS 

In [ ]:
bean = pd.read_csv("../datasets/bean.csv", na_values=na_values)

In [ ]:
print("---------------------------------------")
print("Dataset preview")
print("---------------------------------------")
print(bean.head())
print("---------------------------------------")
print("Columns types")
print("---------------------------------------")
print(bean.dtypes)
print("---------------------------------------")
print("Summary")
print("---------------------------------------")
summary(bean, "Class")

In [ ]:
plt.figure(figsize=(4, 3))
bean['Class'].value_counts(normalize=True).plot(kind="bar")
plt.show()

In [ ]:
numeric_vars = bean.columns[:-1]

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for ax, col in zip(axes, numeric_vars):
    ax.hist(bean[col], bins=20)
    ax.set_title(col)
for ax in axes[len(numeric_vars):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
corr = bean.iloc[:, :-1].corr()

plt.figure(figsize=(6, 5))
cmap = plt.get_cmap("RdBu")   
sns.heatmap(
    corr,
    cmap=cmap,
    vmin=-1, vmax=1,
    center=0,                 # ensures white at 0
    square=True,
    annot=True, fmt=".2f",
    annot_kws={"size": 7},
    linewidths=0.5,
    cbar_kws={"orientation": "vertical"}
)
plt.title("Pearson Correlation (Numerical Variables)")
plt.xticks(rotation=30, ha="right", fontsize=8, color="black")
plt.yticks(rotation=0, fontsize=8, color="black")

plt.tight_layout()
plt.show()

In [ ]:
bean_train, bean_test = train_test_split(bean, test_size=0.2, random_state=42)
print(bean_train.shape)
print(bean_test.shape)

In [ ]:
bean_train.to_csv("../datasets/bean_train.csv", index=False)
bean_test.to_csv("../datasets/bean_test.csv", index=False)

In [ ]:
print("----------------------------------------------------------------------------------------------")
print("Oversampling using RandomOverSampler")
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(bean_train.drop(['Class'], axis = 1), bean_train['Class'])
print(f"Original class distribution: {bean_train['Class'].value_counts()}")
print(f"Resampled class distribution: {pd.Series(y_train_resampled).value_counts()}")

bean_train_resampled = pd.concat(
    [X_train_resampled, y_train_resampled],
    axis=1
)
bean_train_resampled.to_csv("../datasets/bean_train_balanced.csv", index=False)

# CREDIT CARD

In [ ]:
creditcard = pd.read_csv("../datasets/credit_card.csv", na_values=na_values)

In [ ]:
print("---------------------------------------")
print("Dataset preview")
print("---------------------------------------")
print(creditcard.head())
print("---------------------------------------")
print("Columns types")
print("---------------------------------------")
print(creditcard.dtypes)
print("---------------------------------------")
print("Summary")
print("---------------------------------------")
summary(creditcard, "default.payment.next.month")

In [ ]:
plt.figure(figsize=(4, 3))
creditcard['default.payment.next.month'].value_counts(normalize=True).plot(kind="bar")
plt.show()

In [ ]:
categorical_vars = ["SEX", "EDUCATION", "MARRIAGE",
                    "PAY_0", "PAY_2",
                    "PAY_3", "PAY_4",
                    "PAY_5", "PAY_6"]

fig, axes = plt.subplots(3, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, categorical_vars):
    creditcard[col].value_counts(normalize=True).plot(kind="bar", ax=ax)
    ax.set_title(col)
    ax.set_ylim(0, 0.8)

# Hide any unused subplots
for ax in axes[len(categorical_vars):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
numeric_vars = [
    "LIMIT_BAL", "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

fig, axes = plt.subplots(3, 5, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, numeric_vars):
    ax.hist(creditcard[col], bins=20)
    ax.set_title(col)
for ax in axes[len(numeric_vars):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
corr = creditcard[numeric_vars].corr()

plt.figure(figsize=(6, 5))
cmap = plt.get_cmap("RdBu")   
sns.heatmap(
    corr,
    cmap=cmap,
    vmin=-1, vmax=1,
    center=0,                 # ensures white at 0
    square=True,
    annot=True, fmt=".2f",
    annot_kws={"size": 7},
    linewidths=0.5,
    cbar_kws={"orientation": "vertical"}
)
plt.title("Pearson Correlation (Numerical Variables)")
plt.xticks(rotation=30, ha="right", fontsize=8, color="black")
plt.yticks(rotation=0, fontsize=8, color="black")

plt.tight_layout()
plt.show()

In [ ]:
print(creditcard['MARRIAGE'].value_counts()) 
print(creditcard['EDUCATION'].value_counts()) 

creditcard['MARRIAGE'] = creditcard['MARRIAGE'].replace(0, 3)
creditcard['MARRIAGE'] = creditcard['MARRIAGE'].replace(0, 3)

creditcard['EDUCATION'] = creditcard['EDUCATION'].replace(0, 6)
creditcard['EDUCATION'] = creditcard['EDUCATION'].replace(0, 6)

print(creditcard['MARRIAGE'].value_counts()) 
print(creditcard['EDUCATION'].value_counts()) 

In [ ]:
creditcard_train, creditcard_test = train_test_split(creditcard, test_size=0.2, random_state=42)
print(creditcard_train.shape)
print(creditcard_test.shape)

In [ ]:
creditcard_train.to_csv("../datasets/creditcard_train.csv", index=False)
creditcard_test.to_csv("../datasets/creditcard_test.csv", index=False)

In [ ]:
print("----------------------------------------------------------------------------------------------")
print("Oversampling using RandomOverSampler")
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(creditcard_train.drop(['default.payment.next.month'], axis = 1), creditcard_train['default.payment.next.month'])
print(f"Original class distribution: {creditcard_train['default.payment.next.month'].value_counts()}")
print(f"Resampled class distribution: {pd.Series(y_train_resampled).value_counts()}")

creditcard_train_resampled = pd.concat(
    [X_train_resampled, y_train_resampled],
    axis=1
)
creditcard_train_resampled.to_csv("../datasets/creditcard_train_balanced.csv", index=False)